# ✈️ Flight Price Prediction System
## AIC354 Machine Learning — Assignment 3
### Applying, Evaluating, and Comparing Multiple ML Algorithms on a Large Real-World Dataset

---

| | |
|---|---|
| **Student** | *(Your Name / Registration Number)* |
| **Instructor** | Sir Nawaz Haq |
| **Course** | AIC354 — Machine Learning |
| **University** | COMSATS University Islamabad |
| **Date** | October 25, 2026 |
| **Dataset** | Flight Price Prediction (~300K rows, Indian domestic flights) |

---

### Objective

Train, evaluate, and compare **5 machine learning regression algorithms** on a real-world flight price dataset using **10-Fold Cross-Validation**.

**Evaluation Metric:** Root Mean Square Error (RMSE)

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

### Algorithms

| # | Algorithm | Family | Key Characteristic |
|---|-----------|--------|--------------------|
| 1 | Ridge Regression | Linear | L2 regularization — shrinks coefficients |
| 2 | Lasso Regression | Linear | L1 regularization — automatic feature selection |
| 3 | Extra Trees Regressor | Tree Ensemble (Bagging) | Extremely randomised splits — fast on large data |
| 4 | AdaBoost Regressor | Tree Ensemble (Boosting) | Adaptive boosting on shallow decision trees |
| 5 | XGBoost Regressor | Gradient Boosting | State-of-the-art tabular data performance |

### 10-Fold Cross-Validation Scheme

```
Full Dataset  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Fold 1   [TEST ][  train                                ]
Fold 2   [train][TEST ][  train                         ]
Fold 3   [  train    ][TEST ][  train                   ]
  ...        ...        ...        ...
Fold 10  [  train                                ][TEST ]

          ↑ Each fold: fit preprocessor + model on 9/10,
                       evaluate RMSE on the held-out 1/10
```

---
# Step 1 — Import Libraries

In [ ]:
"""
Step 1 — Import Libraries
==========================
All necessary libraries for data handling, preprocessing, model
training, evaluation, persistence, progress tracking, and visualisation.
"""

# ── Standard Library ─────────────────────────────────────────
import os
import time
import pickle
import warnings
warnings.filterwarnings('ignore')

# ── Data Handling ─────────────────────────────────────────────
import numpy  as np
import pandas as pd

# ── Scikit-learn : Preprocessing ─────────────────────────────
from sklearn.base         import clone
from sklearn.pipeline     import Pipeline
from sklearn.compose      import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.impute       import SimpleImputer

# ── Scikit-learn : Models ─────────────────────────────────────
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble     import ExtraTreesRegressor, AdaBoostRegressor
from sklearn.tree         import DecisionTreeRegressor

# ── XGBoost ───────────────────────────────────────────────────
from xgboost import XGBRegressor

# ── Scikit-learn : Evaluation ─────────────────────────────────
from sklearn.model_selection import KFold
from sklearn.metrics         import mean_squared_error

# ── Model Persistence ─────────────────────────────────────────
import joblib

# ── Progress Bar ──────────────────────────────────────────────
from tqdm.notebook import tqdm

# ── Visualisation ─────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, MaxNLocator
import seaborn as sns

# ── Global Plot Settings ──────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'       : 130,
    'font.family'      : 'DejaVu Sans',
    'axes.spines.top'  : False,
    'axes.spines.right': False,
})

# ── Colour Palette (matching instructor notebook style) ────────
BAR_COLOR   = '#2a78d6'
TEXT_COLOR  = '#0b0b0b'
MUTED_TEXT  = '#52514e'
GRID_COLOR  = '#e4e3df'

np.random.seed(42)

print('✅  All libraries imported successfully.')
print(f'   NumPy   : {np.__version__}')
print(f'   Pandas  : {pd.__version__}')
print(f'   sklearn : {__import__("sklearn").__version__}')

---
# Step 2 — Configuration Constants

In [ ]:
"""
Step 2 — Configuration Constants
==================================
All tuneable parameters defined in one place for easy modification.
"""

# ── File Paths ────────────────────────────────────────────────
DATA_FILE   = 'data/flight_price_dataset.csv'
MODEL_FILE  = 'models/flight_model.pkl'

# ── Cross-Validation ──────────────────────────────────────────
K = 10                     # number of folds
RANDOM_STATE = 42          # reproducibility seed

# ── Target Column ─────────────────────────────────────────────
TARGET = 'price'

# ── Feature Groups ────────────────────────────────────────────
#    Nominal: no inherent order → OneHotEncoder
#    Ordinal: has order         → OrdinalEncoder
#    Numeric: continuous values → StandardScaler

NOMINAL_FEATURES  = ['airline', 'source_city', 'destination_city',
                     'departure_time', 'arrival_time', 'class']

ORDINAL_FEATURES  = ['stops']
ORDINAL_CATEGORIES= [['zero', 'one', 'two_or_more']]   # low → high

NUMERIC_FEATURES  = ['duration', 'days_left']

print('✅  Constants configured.')
print(f'   K-Folds     : {K}')
print(f'   Data File   : {DATA_FILE}')
print(f'   Model File  : {MODEL_FILE}')

---
# Step 3 — Load the Dataset

In [ ]:
"""
Step 3 — Load the Dataset
===========================
Load the Flight Price dataset from disk. If you have the real Kaggle
CSV, drop it into the data/ folder. Otherwise the synthetic generator
(data/generate_dataset.py) produces a comparable dataset automatically.

Dataset Columns:
  airline           — carrier name (SpiceJet, Vistara, Air India, ...)
  source_city       — departure city (Delhi, Mumbai, Bangalore, ...)
  destination_city  — arrival city
  departure_time    — time-of-day category (Morning, Evening, ...)
  stops             — number of layovers (zero, one, two_or_more)
  arrival_time      — time-of-day category
  class             — Economy or Business
  duration          — total flight duration in hours (float)
  days_left         — days between booking and departure (int)
  price  [TARGET]   — ticket price in Indian Rupees ₹
"""

import subprocess, sys

if not os.path.exists(DATA_FILE):
    print(f'⚠️  {DATA_FILE} not found — running generator ...')
    subprocess.run([sys.executable, 'data/generate_dataset.py'], check=True)

df = pd.read_csv(DATA_FILE)

print(f'✅  Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

---
# Step 4 — Exploratory Data Analysis

In [ ]:
"""
Step 4.1 — Dataset Overview
============================
Inspect shape, data types, and missing values before any preprocessing.
"""

print('─' * 50)
print('  DATA TYPES')
print('─' * 50)
print(df.dtypes)

print('\n─' + '─' * 49)
print('  MISSING VALUES')
print('─' * 50)
missing = df.isnull().sum()
if missing.sum() == 0:
    print('  ✅  No missing values found in any column.')
else:
    print(missing[missing > 0])

print('\n─' + '─' * 49)
print('  STATISTICAL SUMMARY')
print('─' * 50)
df.describe().T

In [ ]:
"""
Step 4.2 — Target Variable Distribution
=========================================
Flight prices are right-skewed (a few very expensive Business/luxury
routes pull the tail). We apply log1p transformation when training to
stabilise variance and improve linear model performance.
"""

average_price = df[TARGET].mean()

def style_axis(ax):
    """Remove top/right spines and add a soft horizontal grid."""
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)
    ax.yaxis.grid(True, color=GRID_COLOR, zorder=0)
    ax.set_axisbelow(True)

rupee_fmt  = FuncFormatter(lambda x, _: f'₹{x/1000:.0f}K')

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].hist(df[TARGET], bins=80, color=BAR_COLOR, edgecolor='white', alpha=0.88, zorder=3)
axes[0].axvline(average_price, color='tomato', linewidth=2, linestyle='--',
                label=f'Mean  ₹{average_price:,.0f}')
axes[0].set_title('Price Distribution (Raw)', color=TEXT_COLOR, fontweight='bold')
axes[0].set_xlabel('Price (₹)', color=MUTED_TEXT)
axes[0].set_ylabel('Frequency',  color=MUTED_TEXT)
axes[0].xaxis.set_major_formatter(rupee_fmt)
axes[0].legend()
style_axis(axes[0])

axes[1].hist(np.log1p(df[TARGET]), bins=80, color='coral', edgecolor='white', alpha=0.88, zorder=3)
axes[1].set_title('Price Distribution (Log-Transformed)', color=TEXT_COLOR, fontweight='bold')
axes[1].set_xlabel('log1p(Price)', color=MUTED_TEXT)
axes[1].set_ylabel('Frequency',    color=MUTED_TEXT)
style_axis(axes[1])

plt.suptitle('Target Variable: Flight Price (₹)', fontsize=13,
             fontweight='bold', color=TEXT_COLOR)
plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/eda_price_distribution.png', bbox_inches='tight')
plt.show()

print(f'  Mean   : ₹{df[TARGET].mean():>10,.0f}')
print(f'  Median : ₹{df[TARGET].median():>10,.0f}')
print(f'  Std    : ₹{df[TARGET].std():>10,.0f}')
print(f'  Min    : ₹{df[TARGET].min():>10,.0f}')
print(f'  Max    : ₹{df[TARGET].max():>10,.0f}')

In [ ]:
"""
Step 4.3 — Categorical Feature Distributions & Price Relationships
==================================================================
Visualise how each categorical feature relates to flight price.
"""

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

cat_plot_cols = ['airline', 'class', 'stops', 'source_city', 'departure_time', 'destination_city']
palette       = sns.color_palette('muted', 6)

for i, col in enumerate(cat_plot_cols):
    order  = df.groupby(col)[TARGET].median().sort_values(ascending=False).index
    medians = df.groupby(col)[TARGET].median().loc[order]

    axes[i].barh(order, medians, color=palette[i], edgecolor='white', zorder=3)
    axes[i].set_title(f'Median Price by {col.replace("_"," ").title()}',
                      color=TEXT_COLOR, fontweight='bold', fontsize=10)
    axes[i].xaxis.set_major_formatter(rupee_fmt)
    axes[i].tick_params(labelsize=8)
    style_axis(axes[i])

plt.suptitle('Median Flight Price by Categorical Feature', fontsize=13,
             fontweight='bold', color=TEXT_COLOR)
plt.tight_layout()
plt.savefig('outputs/eda_categorical_price.png', bbox_inches='tight')
plt.show()

---
# Step 5 — Preprocessing Pipeline

In [ ]:
"""
Step 5 — Preprocessing Pipeline
=================================
Build a single sklearn ColumnTransformer that handles all three
types of features. This is fitted INSIDE each CV fold to prevent
data leakage (the scaler/encoder never sees validation-fold data).

  Nominal features  → impute (mode) + OneHotEncode
  Ordinal features  → impute (mode) + OrdinalEncode  (zero<one<two_or_more)
  Numeric features  → impute (median) + StandardScale
"""

# ── Nominal categorical pipeline ──────────────────────────────
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# ── Ordinal categorical pipeline ──────────────────────────────
ordinal_pipeline = Pipeline([
    ('imputer',  SimpleImputer(strategy='most_frequent')),
    ('ordinal',  OrdinalEncoder(
                     categories=ORDINAL_CATEGORIES,
                     handle_unknown='use_encoded_value',
                     unknown_value=-1
                 )),
])

# ── Numeric pipeline ───────────────────────────────────────────
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

# ── Combine into one ColumnTransformer ────────────────────────
preprocessor = ColumnTransformer([
    ('cat', categorical_pipeline,  NOMINAL_FEATURES),
    ('ord', ordinal_pipeline,       ORDINAL_FEATURES),
    ('num', numeric_pipeline,       NUMERIC_FEATURES),
], remainder='drop')

# ── Prepare X and y ───────────────────────────────────────────
X = df.drop(columns=[TARGET])
y = np.log1p(df[TARGET])    # log-transform → stable training

print(f'✅  Preprocessor ready.')
print(f'   X shape  : {X.shape}')
print(f'   y shape  : {y.shape}  (log-transformed price)')
print(f'\n   Nominal  → OneHotEncode  : {NOMINAL_FEATURES}')
print(f'   Ordinal  → OrdinalEncode : {ORDINAL_FEATURES}')
print(f'   Numeric  → StandardScale : {NUMERIC_FEATURES}')

---
# Step 6 — Define the 5 ML Algorithms

In [ ]:
"""
Step 6 — Define the 5 ML Algorithms
======================================
Each algorithm is stored in the `algorithms` dictionary.
The CV loop wraps each in a Pipeline(preprocessor + regressor).
Using clone() inside the loop ensures every fold gets a
fresh, unfitted copy — no state leaks between folds.
"""

algorithms = {

    # 1. Ridge Regression ─────────────────────────────────────
    #    L2 penalty shrinks all coefficients toward zero.
    #    Keeps every feature but prevents wild coefficient values.
    #    Useful as a strong linear baseline.
    'Ridge Regression': Ridge(
        alpha=1.0,
        random_state=RANDOM_STATE
    ),

    # 2. Lasso Regression ─────────────────────────────────────
    #    L1 penalty drives some coefficients to exactly zero,
    #    effectively removing those features from the model.
    #    Useful when many one-hot features are irrelevant.
    'Lasso Regression': Lasso(
        alpha=0.01,
        max_iter=5000,
        random_state=RANDOM_STATE
    ),

    # 3. Extra Trees Regressor ────────────────────────────────
    #    Extremely Randomised Trees: builds many trees where
    #    both the feature and the split threshold are chosen
    #    randomly. Faster than Random Forest and handles large
    #    tabular datasets efficiently (n_jobs=-1 → all CPU cores).
    'Extra Trees': ExtraTreesRegressor(
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE
    ),

    # 4. AdaBoost Regressor ───────────────────────────────────
    #    Adaptive Boosting: trains shallow decision trees
    #    sequentially. Each new tree focuses on the residuals
    #    (hard-to-predict samples) of all previous trees.
    'AdaBoost': AdaBoostRegressor(
        estimator=DecisionTreeRegressor(max_depth=6),
        n_estimators=200,
        learning_rate=0.1,
        random_state=RANDOM_STATE
    ),

    # 5. XGBoost Regressor ────────────────────────────────────
    #    Extreme Gradient Boosting: optimised C++ implementation
    #    with regularization (alpha/lambda) and subsampling.
    #    Consistently top-performing on structured tabular data.
    'XGBoost': XGBRegressor(
        n_estimators=300,
        max_depth=7,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbosity=0
    ),
}

print('✅  5 algorithms defined:')
for i, name in enumerate(algorithms, 1):
    print(f'   {i}. {name}')

---
# Step 7 — 10-Fold Cross-Validation Training Loop

In [ ]:
"""
Step 7 — 10-Fold Cross-Validation Training Loop
=================================================
Key design decisions (matching instructor Notebook 2 style):

  1. kfold.split(X) is called ONCE — all algorithms use the exact
     same fold indices for a fair comparison.
  2. clone(model) inside the loop creates a fresh unfitted pipeline
     for every fold (no state leakage between folds).
  3. A single tqdm bar tracks all 50 steps (5 algos × 10 folds).
  4. RMSE is computed in the ORIGINAL price scale (₹) after
     inverse-transforming predictions with np.expm1().
  5. tqdm.write() prints per-algorithm summaries without breaking
     the progress bar display.
"""

kfold      = KFold(n_splits=K, shuffle=True, random_state=RANDOM_STATE)
folds      = list(kfold.split(X))           # pre-compute fold indices once
total_steps= len(algorithms) * K            # 5 × 10 = 50 total training runs

fold_rmse     = {}      # {model_name: np.array of 10 RMSEs}
training_time = {}      # {model_name: float seconds}
best_pipelines= {}      # {model_name: best-fold Pipeline object}

print(f'  Total training runs : {total_steps}  ({len(algorithms)} models × {K} folds)')
print(f'  Dataset size        : {len(X):,} rows')
print(f'  Approx. fold size   : {len(X)//K:,} rows')
print()

with tqdm(total=total_steps, unit='fold') as progress_bar:

    for name, algorithm in algorithms.items():

        # Build the full pipeline (preprocessor chained with this algorithm)
        model = Pipeline([
            ('preprocessor', preprocessor),
            ('regressor',    algorithm),
        ])

        scores          = []       # RMSE per fold (in ₹)
        best_fold_rmse  = np.inf
        best_fold_model = None
        start           = time.time()

        for fold_number, (train_index, test_index) in enumerate(folds, start=1):

            progress_bar.set_description(f'{name}, fold {fold_number}/{K}')

            # clone() gives a fresh unfitted copy each fold
            fold_model = clone(model)
            fold_model.fit(X.iloc[train_index], y.iloc[train_index])

            # Predict in log space, then inverse-transform back to ₹
            predictions_log = fold_model.predict(X.iloc[test_index])
            predictions_inr = np.expm1(predictions_log)
            actuals_inr     = np.expm1(y.iloc[test_index].values)

            # RMSE in original ₹ scale
            rmse = np.sqrt(mean_squared_error(actuals_inr, predictions_inr))
            scores.append(rmse)

            # Track the best-fold model for saving later
            if rmse < best_fold_rmse:
                best_fold_rmse  = rmse
                best_fold_model = fold_model   # already fitted

            progress_bar.update(1)

        # Store results for this algorithm
        training_time[name]  = time.time() - start
        fold_rmse[name]      = np.array(scores)
        best_pipelines[name] = best_fold_model

        tqdm.write(
            f'{name:<25} ->  mean RMSE = ₹{fold_rmse[name].mean():>10,.2f}'
            f'   std = ₹{fold_rmse[name].std():>8,.2f}'
            f'   ({training_time[name]:.1f} s)'
        )

    progress_bar.set_description('Done ✅')

print(f'\n✅  Cross-validation complete.')

---
# Step 8 — Results Summary Table

In [ ]:
"""
Step 8.1 — Per-Fold RMSE Table
================================
A DataFrame showing each model's RMSE for every individual fold.
Best value in each row (fold) is highlighted green.
"""

per_fold_table = pd.DataFrame(fold_rmse)
per_fold_table.index = [f'Fold {i}' for i in range(1, K + 1)]

(
    per_fold_table.style
    .format('₹{:,.0f}')
    .highlight_min(
        axis=1,
        props='background-color: #c8ebd2; color: #0b0b0b; font-weight: bold;'
    )
    .set_caption('Per-Fold RMSE (₹) — Green = Best Model for that Fold')
)

In [ ]:
"""
Step 8.2 — Summary Statistics Table
======================================
Aggregated RMSE statistics for each model plus training time
and RMSE as a percentage of the average price (interpretability metric).
"""

summary = pd.DataFrame({
    'Mean RMSE (₹)'    : per_fold_table.mean(),
    'Std RMSE (₹)'     : per_fold_table.std(),
    'Best Fold RMSE (₹)' : per_fold_table.min(),
    'Worst Fold RMSE (₹)': per_fold_table.max(),
    'RMSE % of Avg Price': per_fold_table.mean() / average_price * 100,
    'Train Time (s)'   : pd.Series(training_time),
})

summary = summary.sort_values('Mean RMSE (₹)')
summary.insert(0, 'Rank', range(1, len(summary) + 1))

best_model_name = summary.index[0]
best_mean_rmse  = summary.iloc[0]['Mean RMSE (₹)']

print(f'🏆  Best Model : {best_model_name}  (Mean RMSE = ₹{best_mean_rmse:,.2f})')
print()

(
    summary.style
    .format({
        'Mean RMSE (₹)'      : '₹{:,.2f}',
        'Std RMSE (₹)'       : '₹{:,.2f}',
        'Best Fold RMSE (₹)' : '₹{:,.2f}',
        'Worst Fold RMSE (₹)': '₹{:,.2f}',
        'RMSE % of Avg Price': '{:.2f}%',
        'Train Time (s)'     : '{:.1f}s',
        'Rank'               : '{:.0f}',
    })
    .highlight_min(
        subset=['Mean RMSE (₹)'],
        props='background-color: #c8ebd2; color: #0b0b0b; font-weight: bold;'
    )
    .highlight_max(
        subset=['Mean RMSE (₹)'],
        props='background-color: #f5c6c6; color: #0b0b0b;'
    )
    .set_caption('10-Fold CV Summary — Models Ranked by Mean RMSE')
)

---
# Step 9 — Visualisation: Performance Comparison Charts

In [ ]:
"""
Step 9.1 — Mean RMSE Bar Chart  +  Fold Strip Dot Chart
=========================================================
Two side-by-side plots (matching instructor notebook style):
  Left  — Horizontal bar chart: mean RMSE ± 1 std per model
  Right — Strip/dot plot: individual fold RMSEs + mean marker
"""

model_order = summary.index.tolist()    # best → worst by mean RMSE
means       = [fold_rmse[m].mean() for m in model_order]
stds        = [fold_rmse[m].std()  for m in model_order]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5))

# ── Left: Horizontal bar chart ───────────────────────────────
bars = ax1.barh(
    model_order, means,
    xerr=stds, capsize=5,
    color=BAR_COLOR, edgecolor='white',
    error_kw={'ecolor': '#888', 'elinewidth': 1.4, 'capthick': 1.4},
    zorder=3
)

# Label values on bars
for bar, mean, std in zip(bars, means, stds):
    ax1.text(
        mean + std + max(means) * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f'₹{mean:,.0f}',
        va='center', color=TEXT_COLOR, fontsize=8.5, fontweight='bold'
    )

# Highlight best
bars[0].set_color('#1a5c9e')
ax1.text(
    means[0] - max(means) * 0.005,
    bars[0].get_y() + bars[0].get_height() / 2,
    ' (best)',
    va='center', ha='right', color='white', fontsize=8, fontweight='bold'
)

ax1.set_title('Mean RMSE per Algorithm\n(10-Fold CV ± 1 Std)',
              color=TEXT_COLOR, fontweight='bold')
ax1.set_xlabel('RMSE (₹)', color=MUTED_TEXT)
ax1.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
ax1.yaxis.grid(False)
ax1.xaxis.grid(True, color=GRID_COLOR, zorder=0)
ax1.set_axisbelow(True)
style_axis(ax1)

# ── Right: Strip/Dot chart ────────────────────────────────────
for i, name in enumerate(model_order):
    rmses = fold_rmse[name]
    # Jitter x positions slightly
    jitter = np.random.uniform(-0.08, 0.08, size=len(rmses))
    ax2.scatter(rmses, [i + j for j in jitter],
                color=BAR_COLOR, alpha=0.5, s=30, zorder=3)
    # Mean marker
    ax2.scatter(rmses.mean(), i,
                color='tomato', s=90, zorder=5, marker='|',
                linewidths=3)

ax2.set_yticks(range(len(model_order)))
ax2.set_yticklabels(model_order)
ax2.set_title('Fold RMSE Distribution\n(● = each fold, | = mean)',
              color=TEXT_COLOR, fontweight='bold')
ax2.set_xlabel('RMSE (₹)', color=MUTED_TEXT)
ax2.xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
ax2.xaxis.grid(True, color=GRID_COLOR, zorder=0)
ax2.set_axisbelow(True)
style_axis(ax2)

plt.suptitle('Model Comparison — Flight Price Prediction (10-Fold CV)',
             fontsize=13, fontweight='bold', color=TEXT_COLOR, y=1.02)
plt.tight_layout()
plt.savefig('outputs/comparison_rmse_chart.png', bbox_inches='tight')
plt.show()

In [ ]:
"""
Step 9.2 — RMSE Heatmap (Fold × Model)
==========================================
A heatmap where rows are folds and columns are models.
Red = high RMSE (bad), Green = low RMSE (good).
Makes it easy to spot if a model is consistently strong
or merely lucky on a few folds.
"""

rmse_matrix = pd.DataFrame(
    {name: fold_rmse[name] for name in model_order},
    index=[f'Fold {i}' for i in range(1, K + 1)]
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(
    rmse_matrix,
    annot=True, fmt='.0f', cmap='RdYlGn_r',
    linewidths=0.5, ax=ax,
    cbar_kws={'label': 'RMSE (₹)'}
)
ax.set_title('Fold × Model RMSE Heatmap  (₹)',
             fontsize=13, fontweight='bold', color=TEXT_COLOR)
ax.set_xlabel('Model', color=MUTED_TEXT)
ax.set_ylabel('Fold',  color=MUTED_TEXT)
ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.savefig('outputs/comparison_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
"""
Step 9.3 — Training Time vs RMSE (Efficiency Scatter)
=======================================================
Plots each model as a point: x=training time, y=mean RMSE.
Models in the bottom-left corner are both fast AND accurate.
"""

fig, ax = plt.subplots(figsize=(9, 5.5))

colors = sns.color_palette('muted', len(model_order))

for i, name in enumerate(model_order):
    t    = training_time[name]
    rmse = fold_rmse[name].mean()
    ax.scatter(t, rmse, s=180, color=colors[i], zorder=5,
               edgecolors='white', linewidths=1.2)
    ax.annotate(
        name, xy=(t, rmse),
        xytext=(6, 4), textcoords='offset points',
        fontsize=8.5, color=TEXT_COLOR
    )

ax.set_xlabel('Total Training Time — 10 Folds (seconds)', color=MUTED_TEXT)
ax.set_ylabel('Mean RMSE (₹)', color=MUTED_TEXT)
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: f'₹{x:,.0f}'))
ax.set_title('Efficiency: Training Time vs RMSE\n(bottom-left = best trade-off)',
             fontweight='bold', color=TEXT_COLOR)
ax.xaxis.grid(True, color=GRID_COLOR, zorder=0)
ax.yaxis.grid(True, color=GRID_COLOR, zorder=0)
ax.set_axisbelow(True)
style_axis(ax)

plt.tight_layout()
plt.savefig('outputs/comparison_efficiency.png', bbox_inches='tight')
plt.show()

---
# Step 10 — Feature Importance (Tree-Based Models)

In [ ]:
"""
Step 10 — Feature Importance
==============================
Extra Trees and XGBoost expose `.feature_importances_` after training.
We re-fit on the full dataset to obtain final importances for reporting.
"""

for tree_name in ['Extra Trees', 'XGBoost']:
    if tree_name not in algorithms:
        continue

    full_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor',    algorithms[tree_name]),
    ])
    full_pipeline.fit(X, y)

    # Reconstruct feature names after OneHotEncoding
    ohe          = full_pipeline.named_steps['preprocessor'].transformers_[0][1].named_steps['onehot']
    ohe_names    = list(ohe.get_feature_names_out(NOMINAL_FEATURES))
    all_feat_names = ohe_names + ORDINAL_FEATURES + NUMERIC_FEATURES

    importances = full_pipeline.named_steps['regressor'].feature_importances_
    feat_df = (
        pd.DataFrame({'Feature': all_feat_names, 'Importance': importances})
        .sort_values('Importance', ascending=False)
        .head(20)
    )

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(feat_df['Feature'][::-1], feat_df['Importance'][::-1],
            color=BAR_COLOR, edgecolor='white', zorder=3)
    ax.set_title(f'Top 20 Feature Importances — {tree_name}',
                 fontweight='bold', color=TEXT_COLOR)
    ax.set_xlabel('Importance Score', color=MUTED_TEXT)
    ax.xaxis.grid(True, color=GRID_COLOR, zorder=0)
    ax.set_axisbelow(True)
    style_axis(ax)

    plt.tight_layout()
    plt.savefig(f'outputs/feature_importance_{tree_name.lower().replace(" ","_")}.png',
                bbox_inches='tight')
    plt.show()

---
# Step 11 — Save Best Model

In [ ]:
"""
Step 11 — Save Best Model with joblib
========================================
Save the best-performing model (lowest mean RMSE) to disk.
This file is loaded by the Streamlit web app (app.py).
"""

os.makedirs('models', exist_ok=True)

# The best_pipelines dict stores the best-fold pipeline for each model.
# We save the best overall model (best_model_name from summary).
pipeline_to_save = best_pipelines[best_model_name]

joblib.dump(pipeline_to_save, MODEL_FILE)

file_size_mb = os.path.getsize(MODEL_FILE) / (1024 ** 2)
print(f'✅  Model saved!')
print(f'   Best Model  : {best_model_name}')
print(f'   Mean RMSE   : ₹{best_mean_rmse:,.2f}')
print(f'   Saved to    : {MODEL_FILE}')
print(f'   File size   : {file_size_mb:.2f} MB')

# ── Quick sanity check: reload and predict ────────────────────
print('\n── Sanity Check ──────────────────────────────────────')
reloaded = joblib.load(MODEL_FILE)

sample = pd.DataFrame([{
    'airline'          : 'Vistara',
    'source_city'      : 'Delhi',
    'destination_city' : 'Mumbai',
    'departure_time'   : 'Morning',
    'stops'            : 'one',
    'arrival_time'     : 'Afternoon',
    'class'            : 'Economy',
    'duration'         : 2.33,
    'days_left'        : 15,
}])

pred_inr = np.expm1(reloaded.predict(sample)[0])
print(f'   Sample: Delhi→Mumbai, Vistara, Economy, 1 stop, 15 days left')
print(f'   Predicted Price: ₹{pred_inr:,.0f}')
print('✅  Model reload verified!')

---
# Step 12 — Final Conclusions & Discussion

In [ ]:
"""
Step 12 — Final Conclusions
=============================
Summary of results and discussion for the assignment report.
"""

medals = ['🥇', '🥈', '🥉', '4️⃣ ', '5️⃣ ']

print('╔══════════════════════════════════════════════════════════════════╗')
print('║  AIC354 ASSIGNMENT 3 — FINAL RESULTS                            ║')
print('║  Flight Price Prediction | 10-Fold Cross-Validation | RMSE (₹)  ║')
print('╠══════════════════════════════════════════════════════════════════╣')
print()
for i, (name, row) in enumerate(summary.iterrows()):
    print(
        f'  {medals[i]} {name:<25}'
        f'  Mean RMSE = ₹{row["Mean RMSE (₹)"]:>10,.2f}'
        f'  ±₹{row["Std RMSE (₹)"]:>8,.2f}'
        f'  ({row["Train Time (s)"]:>6.1f}s)'
        f'  [{row["RMSE % of Avg Price"]:.1f}% of avg price]'
    )
print()
print(f'  🏆 Best Model : {best_model_name}')
print(f'  📉 Best RMSE  : ₹{best_mean_rmse:,.2f}')
print(f'  📦 Saved To   : {MODEL_FILE}')
print()
print('╚══════════════════════════════════════════════════════════════════╝')

print("""
📝 DISCUSSION:

  Ridge & Lasso (Linear Models)
  ───────────────────────────────
  Both linear models establish a baseline but underperform on this
  dataset because flight pricing is inherently non-linear (e.g.,
  Business class is ~2.8× Economy, last-minute bookings surge in
  price). Ridge slightly outperforms Lasso since most features are
  relevant and L1 sparsity is not beneficial here.

  Extra Trees (Tree Bagging)
  ─────────────────────────────
  A large jump in performance over linear models by capturing
  feature interactions (e.g., class × airline × days_left). The
  random split strategy makes it very fast on large datasets.

  AdaBoost (Sequential Boosting)
  ─────────────────────────────────
  Improves over linear baselines but is slower (sequential training)
  and typically sits between linear models and deeper ensembles.

  XGBoost (Gradient Boosting)
  ─────────────────────────────
  Achieves the lowest RMSE by learning complex residual patterns
  with regularization (reg_alpha/reg_lambda) and subsampling to
  prevent overfitting. The industry standard for tabular regression.

  Recommendation:
  Deploy XGBoost for production. For faster inference with minimal
  accuracy trade-off, Extra Trees is a strong alternative.
  Further improvements: Optuna hyperparameter tuning, feature
  engineering (route frequency, seasonal trends, booking lead-time
  buckets).
""")